# Scratch Training
### From Stage to System: Bias Propagation in Clinical AI Pipelines
**Luwam Major Kefali**

**Hilina Fissha Woreta**


This notebook extends the Stage 1 sub-analysis by training XGBoost from scratch on the 3,986-patient cohort. Both models are trained on the same data and with the same algorithm; the only difference is that Model B includes the 9 discharge note features as additional inputs. 

## Setup

In [29]:
import numpy as np
import pandas as pd
import json
import xgboost as xgb
import optuna
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.isotonic import IsotonicRegression

optuna.logging.set_verbosity(optuna.logging.WARNING)

random_seed = 42

preprocessing_dir = "/kaggle/input/notebooks/hilinafissha16/preprocessing"
stage1_path       = "/kaggle/input/datasets/hilinafissha16/mimic-notes/stage1_llama_extraction.parquet"
clf_dir           = "/kaggle/input/notebooks/hilinafissha16/readmission-risk-classifier"

with open(f"{clf_dir}/stage2_encodings.json") as f:
    encodings = json.load(f)

RACE_TO_ENC      = encodings["race"]
RACES            = sorted(RACE_TO_ENC, key=RACE_TO_ENC.get)
INS_TO_ENC       = encodings["insurance"]
ADMISSION_TO_ENC = encodings["admission_type"]

print("setup complete")


setup complete


## Loading the data

Stacking all three splits and filtering to the patients we have discharge note features for.

In [30]:
splits = []
for split_name in ["train", "val", "test"]:
    df = pd.read_parquet(f"{preprocessing_dir}/splits/{split_name}.parquet")
    df["split"] = split_name
    splits.append(df)

all_data = pd.concat(splits, ignore_index=True)

stage1 = pd.read_parquet(stage1_path)

matched = all_data[all_data["hadm_id"].isin(stage1["hadm_id"])].copy()
matched = matched.reset_index(drop=True)
matched = matched.merge(stage1, on="hadm_id", how="left")

n_before_success_filter = len(matched)
failed = matched[matched["extraction_success"] == False]
cohort = matched[matched["extraction_success"] == True].copy().reset_index(drop=True)

print("patients matched to a Stage 1 note:", n_before_success_filter)
print("dropped for a failed extraction:   ", len(failed))
print("kept (extraction_success == True): ", len(cohort))

if len(failed) > 0:
    print("\nrace distribution, dropped for failed extraction (%):")
    print((failed["race_clean"].value_counts(normalize=True) * 100).round(1))
    print("\nrace distribution, kept (%):")
    print((cohort["race_clean"].value_counts(normalize=True) * 100).round(1))

print("\ncohort size:", len(cohort))
print("race distribution:")
print((cohort["race_clean"].value_counts(normalize=True) * 100).round(1))
print("readmission rate:", round(cohort["readmitted_30d"].mean() * 100, 1), "%")

patients matched to a Stage 1 note: 4000
dropped for a failed extraction:    5
kept (extraction_success == True):  3995

race distribution, dropped for failed extraction (%):
race_clean
White    100.0
Name: proportion, dtype: float64

race distribution, kept (%):
race_clean
White                     70.0
Black/African American    14.5
Other/Unknown              7.4
Hispanic/Latino            5.0
Asian                      3.1
Name: proportion, dtype: float64

cohort size: 3995
race distribution:
race_clean
White                     70.0
Black/African American    14.5
Other/Unknown              7.4
Hispanic/Latino            5.0
Asian                      3.1
Name: proportion, dtype: float64
readmission rate: 21.6 %


## LLM feature variance check

Confirming that the discharge note features have real variance before training. Near-constant features would be uninformative and could indicate extraction problems.

In [31]:
llm_check_cols = [
    "comorbidity_burden_score",
    "discharge_risk_indicator_count",
    "sdoh_housing_instability",
    "sdoh_food_insecurity",
    "sdoh_substance_use",
    "sdoh_limited_social_support",
    "sdoh_unemployment",
    "sdoh_transportation_barrier",
]

print("LLM feature means:")
print(cohort[llm_check_cols].mean().round(3))

print("\nvalue counts for binary SDOH features:")
for col in [c for c in llm_check_cols if "sdoh_" in c]:
    print(f"  {col}: {cohort[col].value_counts().to_dict()}")

LLM feature means:
comorbidity_burden_score          4.823
discharge_risk_indicator_count    2.349
sdoh_housing_instability          0.030
sdoh_food_insecurity              0.012
sdoh_substance_use                0.035
sdoh_limited_social_support       0.012
sdoh_unemployment                 0.004
sdoh_transportation_barrier       0.003
dtype: float64

value counts for binary SDOH features:
  sdoh_housing_instability: {0: 3874, 1: 121}
  sdoh_food_insecurity: {0: 3947, 1: 48}
  sdoh_substance_use: {0: 3857, 1: 138}
  sdoh_limited_social_support: {0: 3946, 1: 49}
  sdoh_unemployment: {0: 3980, 1: 15}
  sdoh_transportation_barrier: {0: 3983, 1: 12}


## Discharge note columns: what we use and what we drop

The Stage 1 LLM extraction produces 12 columns per patient. Not all of them are usable as model features.

**Used as features (9 columns):**
- `comorbidity_burden_score`: a 0-10 numeric score summarising how many and how severe the patient's chronic conditions are, as described in the discharge note
- `psychiatric_complexity`: a string label (low / medium / high) indicating the complexity of the patient's psychiatric history, re-encoded as a numeric scale (0/1/2) before training
- `discharge_risk_indicator_count`: a count of how many distinct readmission risk factors were mentioned in the note
- `sdoh_housing_instability`, `sdoh_food_insecurity`, `sdoh_substance_use`, `sdoh_limited_social_support`, `sdoh_unemployment`, `sdoh_transportation_barrier`: six binary flags, each 1 if the corresponding social determinant of health was mentioned in the note and 0 otherwise

**Dropped (3 columns):**
- `hadm_id`: the join key used to match patients across datasets, not a clinical signal
- `sdoh_evidence`: a free-text field containing the raw sentence excerpts the LLM used to justify its SDOH flags, not a structured input that a tree model can use
- `discharge_risk_indicators_text`: similarly, a free-text list of the risk phrases extracted from the note, dropped for the same reason

The two text columns capture potentially useful clinical language but cannot be passed directly to XGBoost. They are kept in the dataframe for reference but excluded from the feature set.

## Feature preparation

Same feature set as the main classifier notebook, with the same encoding approach. For Model B we add the 9 discharge note features on top.


In [32]:
cols_to_drop = [
    "subject_id", "hadm_id", "admittime", "dischtime",
    "readmitted_30d", "race_x_sex", "race_x_insurance",
    "race_clean", "insurance_clean", "sex", "admission_type",
    "hospital_expire_flag", "split",
    "psychiatric_complexity", "sdoh_evidence", "discharge_risk_indicators_text",
    "extraction_success", "plausible",
]

cohort["sex_enc"]            = cohort["sex"].map({"Male": 0, "Female": 1}).fillna(-1)
cohort["race_enc"]           = cohort["race_clean"].map(RACE_TO_ENC).fillna(-1).astype(int)
cohort["insurance_enc"]      = cohort["insurance_clean"].map(INS_TO_ENC).fillna(-1).astype(int)
cohort["admission_type_enc"] = cohort["admission_type"].map(ADMISSION_TO_ENC).fillna(-1).astype(int)

psych_map = {"low": 0, "medium": 1, "high": 2, "unknown": -1}
cohort["psychiatric_complexity_enc"] = cohort["psychiatric_complexity"].map(psych_map).fillna(-1)

llm_feature_cols = [
    "comorbidity_burden_score",
    "psychiatric_complexity_enc",
    "discharge_risk_indicator_count",
    "sdoh_housing_instability",
    "sdoh_food_insecurity",
    "sdoh_substance_use",
    "sdoh_limited_social_support",
    "sdoh_unemployment",
    "sdoh_transportation_barrier",
]

tabular_feature_cols = [c for c in cohort.columns if c not in cols_to_drop
                        and c not in llm_feature_cols]
tabular_feature_cols = list(dict.fromkeys(tabular_feature_cols))

print("tabular features:", len(tabular_feature_cols))
print("llm features:", len(llm_feature_cols))
print("total for model B:", len(tabular_feature_cols) + len(llm_feature_cols))

EXPECTED_TABULAR_COUNT = 35
if len(tabular_feature_cols) != EXPECTED_TABULAR_COUNT:
    raise ValueError(
        f"expected {EXPECTED_TABULAR_COUNT} tabular features to match the main classifier, "
        f"got {len(tabular_feature_cols)}. Model A would no longer be a clean tabular-only "
        f"control if this drifts. Extra/unexpected columns: {tabular_feature_cols}"
    )
print("feature count check passed")

llm_nan_counts = cohort[llm_feature_cols].isna().sum()
if llm_nan_counts.sum() > 0:
    print("\nWARNING -- LLM features still have missing values after filtering to extraction_success == True:")
    print(llm_nan_counts[llm_nan_counts > 0].to_string())
else:
    print("no missing values in LLM feature columns")

SEXES      = sorted(cohort["sex"].dropna().unique().tolist())
SEX_TO_ENC = {"Male": 0, "Female": 1}
INS_GROUPS = sorted(INS_TO_ENC, key=INS_TO_ENC.get)

tabular features: 35
llm features: 9
total for model B: 44
feature count check passed
no missing values in LLM feature columns


## Train / val / test split

Splitting 70/15/15. The val set is used for early stopping during XGBoost training and for setting the enrollment threshold. The test set is held out.

In [33]:
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=random_seed)
train_idx, temp_idx = next(gss1.split(cohort, groups=cohort["subject_id"]))

temp_cohort = cohort.iloc[temp_idx]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=random_seed)
val_rel_idx, test_rel_idx = next(gss2.split(temp_cohort, groups=temp_cohort["subject_id"]))

train_cohort = cohort.iloc[train_idx].reset_index(drop=True)
val_cohort   = temp_cohort.iloc[val_rel_idx].reset_index(drop=True)
test_cohort  = temp_cohort.iloc[test_rel_idx].reset_index(drop=True)

s_tr = set(train_cohort["subject_id"])
s_va = set(val_cohort["subject_id"])
s_te = set(test_cohort["subject_id"])
assert not (s_tr & s_va) and not (s_tr & s_te) and not (s_va & s_te), "patient overlap between splits!"
print("patient overlap check passed")

target = "readmitted_30d"

y_train = train_cohort[target]
y_val   = val_cohort[target]
y_test  = test_cohort[target]

print("train:", len(train_cohort), "| readmission rate:", round(y_train.mean() * 100, 1), "%")
print("val:  ", len(val_cohort),   "| readmission rate:", round(y_val.mean() * 100, 1), "%")
print("test: ", len(test_cohort),  "| readmission rate:", round(y_test.mean() * 100, 1), "%")


patient overlap check passed
train: 2794 | readmission rate: 21.1 %
val:   597 | readmission rate: 22.4 %
test:  604 | readmission rate: 23.2 %


## Fairness metric helper

Same function used throughout the project.

In [34]:
def compute_fairness_metrics(df, group_col, enrolled_col, outcome_col):
    results = []
    for group in df[group_col].unique():
        subset      = df[df[group_col] == group]
        readmitted  = subset[subset[outcome_col] == 1]
        not_readmit = subset[subset[outcome_col] == 0]
        results.append({
            "group":           group,
            "n":               len(subset),
            "enrollment_rate": round(subset[enrolled_col].mean(), 3),
            "tpr":             round(readmitted[enrolled_col].mean(), 3) if len(readmitted) > 0 else 0,
            "fpr":             round(not_readmit[enrolled_col].mean(), 3) if len(not_readmit) > 0 else 0,
        })
    out = pd.DataFrame(results).sort_values("enrollment_rate", ascending=False)
    rates = out["enrollment_rate"]
    dp_ratio = round(rates.min() / rates.max(), 3) if rates.max() > 0 else 0
    return out, dp_ratio

In [35]:
def compute_cfvr_attr(score_fn, df, threshold, group_col, enc_col, groups, enc_map):
    orig_scores   = score_fn(df)
    orig_enrolled = (orig_scores >= threshold).astype(int)
    any_flip = np.zeros(len(df), dtype=int)
    for g in groups:
        mask = (df[group_col].values != g)
        if mask.sum() == 0:
            continue
        cf = df.copy()
        cf.loc[cf.index[mask], group_col] = g
        cf.loc[cf.index[mask], enc_col]   = enc_map[g]
        cf_scores   = score_fn(cf)
        cf_enrolled = (cf_scores >= threshold).astype(int)
        flipped = ((cf_enrolled != orig_enrolled) & mask).astype(int)
        any_flip = np.maximum(any_flip, flipped)
    cfvr_val = round(float(any_flip.mean()), 4)
    breakdown = (
        df[[group_col]].copy()
        .assign(flipped=any_flip)
        .groupby(group_col)["flipped"].mean()
        .round(4).reset_index()
        .rename(columns={"flipped": "cfvr"})
    )
    return cfvr_val, breakdown

def compute_cfvr(score_fn, df, threshold):
    return compute_cfvr_attr(score_fn, df, threshold, "race_clean", "race_enc", RACES, RACE_TO_ENC)

def dp_ratio_from_enrolled(enrolled, group_labels):
    rates = pd.DataFrame({"enrolled": enrolled, "group": group_labels}).groupby("group")["enrolled"].mean()
    return round(float(rates.min() / rates.max()), 3) if rates.max() > 0 else 0.0



## Model A: tabular features only

XGBoost trained from scratch on the 2,790-patient training set using the same 35 tabular features as the main classifier. Optuna tunes the hyperparameters against the val set.

In [36]:
x_train_a = train_cohort[tabular_feature_cols]
x_val_a   = val_cohort[tabular_feature_cols]
x_test_a  = test_cohort[tabular_feature_cols]

def objective_a(trial):
    params = {
        "n_estimators":          trial.suggest_int("n_estimators", 100, 500),
        "learning_rate":         trial.suggest_float("learning_rate", 0.01, 0.1),
        "max_depth":             trial.suggest_int("max_depth", 3, 8),
        "subsample":             trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":      trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight":      trial.suggest_int("min_child_weight", 1, 10),
        "scale_pos_weight":      (y_train == 0).sum() / (y_train == 1).sum(),
        "random_state":          random_seed,
        "eval_metric":           "auc",
        "early_stopping_rounds": 20,
    }
    m = xgb.XGBClassifier(**params, verbosity=0)
    m.fit(x_train_a, y_train, eval_set=[(x_val_a, y_val)], verbose=False)
    return roc_auc_score(y_val, m.predict_proba(x_val_a)[:, 1])

study_a = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=random_seed)
)
study_a.optimize(objective_a, n_trials=30, show_progress_bar=True)

print("best AUROC (val):", round(study_a.best_value, 4))
print("best params:", study_a.best_params)

  0%|          | 0/30 [00:00<?, ?it/s]

best AUROC (val): 0.6443
best params: {'n_estimators': 339, 'learning_rate': 0.09296868115208053, 'max_depth': 3, 'subsample': 0.6783931449676581, 'colsample_bytree': 0.6180909155642152, 'min_child_weight': 4}


In [37]:
model_a = xgb.XGBClassifier(
    **study_a.best_params,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    random_state=random_seed,
    eval_metric="auc",
    early_stopping_rounds=20,
    verbosity=0
)
model_a.fit(x_train_a, y_train, eval_set=[(x_val_a, y_val)], verbose=False)

val_preds_a  = model_a.predict_proba(x_val_a)[:, 1]
test_preds_a = model_a.predict_proba(x_test_a)[:, 1]

iso_a = IsotonicRegression(out_of_bounds="clip")
iso_a.fit(val_preds_a, y_val)
val_cal_a  = iso_a.predict(val_preds_a)
test_cal_a = iso_a.predict(test_preds_a)

threshold_a = pd.Series(val_cal_a).quantile(0.90)

val_enrolled_a = (val_cal_a >= threshold_a).mean()
test_cohort["enrolled_a"] = (test_cal_a >= threshold_a).astype(int)

metrics_a, dp_ratio_a = compute_fairness_metrics(
    test_cohort, "race_clean", "enrolled_a", "readmitted_30d"
)

auroc_a = roc_auc_score(y_test, test_cal_a)

print("Model A (tabular only, trained from scratch)")
print("AUROC:   ", round(auroc_a, 4))
print("dp_ratio:", dp_ratio_a)
print("enrollment rate (val / threshold target):", round(val_enrolled_a * 100, 1), "%")
print("enrollment rate (test / reported):        ", round(test_cohort["enrolled_a"].mean() * 100, 1), "%")
print("\nenrollment by race:")
print(metrics_a.to_string(index=False))

Model A (tabular only, trained from scratch)
AUROC:    0.6587
dp_ratio: 0.123
enrollment rate (val / threshold target): 10.2 %
enrollment rate (test / reported):         12.6 %

enrollment by race:
                 group   n  enrollment_rate   tpr   fpr
       Hispanic/Latino  26            0.308 0.444 0.235
                 Asian  27            0.222 1.000 0.045
Black/African American 103            0.184 0.280 0.154
                 White 422            0.100 0.194 0.071
         Other/Unknown  26            0.038 0.000 0.043


In [38]:
def score_fn_a(df):
    return iso_a.predict(model_a.predict_proba(df[tabular_feature_cols])[:, 1])

cfvr_a, cfvr_a_bd = compute_cfvr(score_fn_a, test_cohort, threshold_a)
print(f"Model A baseline CFVR: {cfvr_a}")
print(cfvr_a_bd.to_string(index=False))


Model A baseline CFVR: 0.0182
            race_clean   cfvr
                 Asian 0.0000
Black/African American 0.0097
       Hispanic/Latino 0.0769
         Other/Unknown 0.0000
                 White 0.0190


## Model B: tabular + discharge note features

Same setup as Model A but with the 9 discharge note features added to the feature set. Optuna tunes separately so both models get the best possible hyperparameters for their respective feature sets.

In [39]:
all_feature_cols = tabular_feature_cols + llm_feature_cols

x_train_b = train_cohort[all_feature_cols]
x_val_b   = val_cohort[all_feature_cols]
x_test_b  = test_cohort[all_feature_cols]

def objective_b(trial):
    params = {
        "n_estimators":          trial.suggest_int("n_estimators", 100, 500),
        "learning_rate":         trial.suggest_float("learning_rate", 0.01, 0.1),
        "max_depth":             trial.suggest_int("max_depth", 3, 8),
        "subsample":             trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":      trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight":      trial.suggest_int("min_child_weight", 1, 10),
        "scale_pos_weight":      (y_train == 0).sum() / (y_train == 1).sum(),
        "random_state":          random_seed,
        "eval_metric":           "auc",
        "early_stopping_rounds": 20,
    }
    m = xgb.XGBClassifier(**params, verbosity=0)
    m.fit(x_train_b, y_train, eval_set=[(x_val_b, y_val)], verbose=False)
    return roc_auc_score(y_val, m.predict_proba(x_val_b)[:, 1])

study_b = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=random_seed)
)
study_b.optimize(objective_b, n_trials=30, show_progress_bar=True)

print("best AUROC (val):", round(study_b.best_value, 4))
print("best params:", study_b.best_params)

  0%|          | 0/30 [00:00<?, ?it/s]

best AUROC (val): 0.638
best params: {'n_estimators': 383, 'learning_rate': 0.09948656929178631, 'max_depth': 5, 'subsample': 0.9620660102743925, 'colsample_bytree': 0.9126654171655951, 'min_child_weight': 3}


In [40]:
model_b = xgb.XGBClassifier(
    **study_b.best_params,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    random_state=random_seed,
    eval_metric="auc",
    early_stopping_rounds=20,
    verbosity=0
)
model_b.fit(x_train_b, y_train, eval_set=[(x_val_b, y_val)], verbose=False)

val_preds_b  = model_b.predict_proba(x_val_b)[:, 1]
test_preds_b = model_b.predict_proba(x_test_b)[:, 1]

iso_b = IsotonicRegression(out_of_bounds="clip")
iso_b.fit(val_preds_b, y_val)
val_cal_b  = iso_b.predict(val_preds_b)
test_cal_b = iso_b.predict(test_preds_b)

threshold_b = pd.Series(val_cal_b).quantile(0.90)

val_enrolled_b = (val_cal_b >= threshold_b).mean()
test_cohort["enrolled_b"] = (test_cal_b >= threshold_b).astype(int)

metrics_b, dp_ratio_b = compute_fairness_metrics(
    test_cohort, "race_clean", "enrolled_b", "readmitted_30d"
)

auroc_b = roc_auc_score(y_test, test_cal_b)

print("Model B (tabular + discharge note features, trained from scratch)")
print("AUROC:   ", round(auroc_b, 4))
print("dp_ratio:", dp_ratio_b)
print("enrollment rate (val / threshold target):", round(val_enrolled_b * 100, 1), "%")
print("enrollment rate (test / reported):        ", round(test_cohort["enrolled_b"].mean() * 100, 1), "%")
print("\nenrollment by race:")
print(metrics_b.to_string(index=False))

Model B (tabular + discharge note features, trained from scratch)
AUROC:    0.6466
dp_ratio: 0.123
enrollment rate (val / threshold target): 13.2 %
enrollment rate (test / reported):         12.7 %

enrollment by race:
                 group   n  enrollment_rate   tpr   fpr
       Hispanic/Latino  26            0.308 0.444 0.235
Black/African American 103            0.194 0.240 0.179
                 Asian  27            0.148 0.800 0.000
                 White 422            0.104 0.204 0.074
         Other/Unknown  26            0.038 0.000 0.043


In [41]:
def score_fn_b(df):
    return iso_b.predict(model_b.predict_proba(df[all_feature_cols].fillna(0))[:, 1])

cfvr_b, cfvr_b_bd = compute_cfvr(score_fn_b, test_cohort, threshold_b)
print(f"Model B baseline CFVR: {cfvr_b}")
print(cfvr_b_bd.to_string(index=False))


Model B baseline CFVR: 0.0017
            race_clean   cfvr
                 Asian 0.0000
Black/African American 0.0097
       Hispanic/Latino 0.0000
         Other/Unknown 0.0000
                 White 0.0000


## Feature importance for Model B

Checking where the discharge note features rank relative to the tabular features. This tells us whether XGBoost actually used them and which ones drove the most signal.

In [42]:
importance_b = pd.DataFrame({
    "feature":    all_feature_cols,
    "importance": model_b.feature_importances_
}).sort_values("importance", ascending=False)

print("top 20 features:")
print(importance_b.head(20).to_string(index=False))

print("\ndischarge note feature importances:")
print(importance_b[importance_b["feature"].isin(llm_feature_cols)].to_string(index=False))

top 20 features:
                       feature  importance
          prior_admissions_12m    0.065812
                     cm_cancer    0.045623
              cm_liver_disease    0.043114
      sdoh_housing_instability    0.036263
             comorbidity_count    0.034940
                    hemoglobin    0.031061
discharge_risk_indicator_count    0.030839
               cm_coagulopathy    0.030108
                      los_days    0.028399
                        cm_cpd    0.027680
                 insurance_enc    0.026355
                   cm_diabetes    0.026161
                           bun    0.026006
    psychiatric_complexity_enc    0.025907
                     platelets    0.025791
      comorbidity_burden_score    0.025523
                 icu_los_total    0.025191
                   n_icu_stays    0.024818
                           wbc    0.022893
                        cm_chf    0.022860

discharge note feature importances:
                       feature  importance


## Comparison

Side by side summary of both models trained from scratch on the same 2,790 patients.

In [43]:
print("=" * 50)
print("SUMMARY: Stage 1 Scratch Training Sub-Analysis")
print(f"Cohort: {len(test_cohort)} patients (held-out test set)")
print("=" * 50)
print(f"{'Metric':<25} {'Model A':>12} {'Model B':>12} {'Change':>12}")
print("-" * 50)
print(f"{'AUROC':<25} {auroc_a:>12.4f} {auroc_b:>12.4f} {auroc_b - auroc_a:>+12.4f}")
print(f"{'dp_ratio':<25} {dp_ratio_a:>12.3f} {dp_ratio_b:>12.3f} {dp_ratio_b - dp_ratio_a:>+12.3f}")
print("=" * 50)

print("\nenrollment rates by race:")
comparison = metrics_a[["group", "n", "enrollment_rate"]].rename(
    columns={"enrollment_rate": "enroll_rate_a"}
).merge(
    metrics_b[["group", "enrollment_rate"]].rename(
        columns={"enrollment_rate": "enroll_rate_b"}
    ),
    on="group"
)
comparison["change"] = (comparison["enroll_rate_b"] - comparison["enroll_rate_a"]).round(3)
print(comparison.to_string(index=False))



SUMMARY: Stage 1 Scratch Training Sub-Analysis
Cohort: 604 patients (held-out test set)
Metric                         Model A      Model B       Change
--------------------------------------------------
AUROC                           0.6587       0.6466      -0.0121
dp_ratio                         0.123        0.123       +0.000

enrollment rates by race:
                 group   n  enroll_rate_a  enroll_rate_b  change
       Hispanic/Latino  26          0.308          0.308   0.000
                 Asian  27          0.222          0.148  -0.074
Black/African American 103          0.184          0.194   0.010
                 White 422          0.100          0.104   0.004
         Other/Unknown  26          0.038          0.038   0.000


## CDA on the Stage 1 Cohort (Sub-Analysis)

Testing whether CDA still helps on this much smaller Stage 1 cohort (3,986 patients vs. 400,000), for Model A and Model B both. This is a scoped-down check, one method instead of the full six-method suite, and by-race CFVR here should be read as suggestive, not precise, since some race groups have only a few dozen patients in the test set.


In [44]:
def build_cda_twins(source_df, races):
    twins = []
    for race in races:
        other_races = [r for r in races if r != race]
        subset = source_df[source_df["race_clean"] == race]
        for other_race in other_races:
            twin = subset.copy()
            twin["race_clean"] = other_race
            twins.append(twin)
    return pd.concat(twins, ignore_index=True)

twins_a = build_cda_twins(train_cohort, RACES)
train_aug_a = pd.concat([train_cohort, twins_a], ignore_index=True)
train_aug_a["race_enc"] = train_aug_a["race_clean"].map(RACE_TO_ENC).fillna(-1).astype(int)

x_train_aug_a = train_aug_a[tabular_feature_cols]
y_train_aug_a = train_aug_a[target]

model_a_cda = xgb.XGBClassifier(
    **study_a.best_params,
    scale_pos_weight=(y_train_aug_a == 0).sum() / (y_train_aug_a == 1).sum(),
    random_state=random_seed,
    eval_metric="auc",
    early_stopping_rounds=20,
    verbosity=0
)
model_a_cda.fit(x_train_aug_a, y_train_aug_a, eval_set=[(x_val_a, y_val)], verbose=False)

val_preds_a_cda  = model_a_cda.predict_proba(x_val_a)[:, 1]
test_preds_a_cda = model_a_cda.predict_proba(x_test_a)[:, 1]

iso_a_cda = IsotonicRegression(out_of_bounds="clip")
iso_a_cda.fit(val_preds_a_cda, y_val)
val_cal_a_cda  = iso_a_cda.predict(val_preds_a_cda)
test_cal_a_cda = iso_a_cda.predict(test_preds_a_cda)

threshold_a_cda = pd.Series(val_cal_a_cda).quantile(0.90)
auroc_a_cda     = roc_auc_score(y_test, test_cal_a_cda)

test_cohort["enrolled_a_cda"] = (test_cal_a_cda >= threshold_a_cda).astype(int)
_, dp_ratio_a_cda = compute_fairness_metrics(
    test_cohort, "race_clean", "enrolled_a_cda", "readmitted_30d"
)

def score_fn_a_cda(df):
    return iso_a_cda.predict(model_a_cda.predict_proba(df[tabular_feature_cols])[:, 1])

cfvr_a_cda, _ = compute_cfvr(score_fn_a_cda, test_cohort, threshold_a_cda)
bmr_a_cda = round((cfvr_a_cda - cfvr_a) / cfvr_a, 4) if cfvr_a > 0 else 0.0

print("Model A + CDA")
print(f"AUROC:    {auroc_a_cda:.4f}  (Model A baseline: {auroc_a:.4f})")
print(f"dp_ratio: {dp_ratio_a_cda:.3f}  (Model A baseline: {dp_ratio_a:.3f})")
print(f"CFVR:     {cfvr_a_cda}  (Model A baseline: {cfvr_a})")
print(f"BMR:      {bmr_a_cda:+.4f}")


Model A + CDA
AUROC:    0.6451  (Model A baseline: 0.6587)
dp_ratio: 0.099  (Model A baseline: 0.123)
CFVR:     0.0  (Model A baseline: 0.0182)
BMR:      -1.0000


In [45]:
twins_b = build_cda_twins(train_cohort, RACES)
train_aug_b = pd.concat([train_cohort, twins_b], ignore_index=True)
train_aug_b["race_enc"] = train_aug_b["race_clean"].map(RACE_TO_ENC).fillna(-1).astype(int)

x_train_aug_b = train_aug_b[all_feature_cols]
y_train_aug_b = train_aug_b[target]

model_b_cda = xgb.XGBClassifier(
    **study_b.best_params,
    scale_pos_weight=(y_train_aug_b == 0).sum() / (y_train_aug_b == 1).sum(),
    random_state=random_seed,
    eval_metric="auc",
    early_stopping_rounds=20,
    verbosity=0
)
model_b_cda.fit(x_train_aug_b, y_train_aug_b, eval_set=[(x_val_b, y_val)], verbose=False)

val_preds_b_cda  = model_b_cda.predict_proba(x_val_b)[:, 1]
test_preds_b_cda = model_b_cda.predict_proba(x_test_b)[:, 1]

iso_b_cda = IsotonicRegression(out_of_bounds="clip")
iso_b_cda.fit(val_preds_b_cda, y_val)
val_cal_b_cda  = iso_b_cda.predict(val_preds_b_cda)
test_cal_b_cda = iso_b_cda.predict(test_preds_b_cda)

threshold_b_cda = pd.Series(val_cal_b_cda).quantile(0.90)
auroc_b_cda     = roc_auc_score(y_test, test_cal_b_cda)

test_cohort["enrolled_b_cda"] = (test_cal_b_cda >= threshold_b_cda).astype(int)
_, dp_ratio_b_cda = compute_fairness_metrics(
    test_cohort, "race_clean", "enrolled_b_cda", "readmitted_30d"
)

def score_fn_b_cda(df):
    return iso_b_cda.predict(model_b_cda.predict_proba(df[all_feature_cols].fillna(0))[:, 1])

cfvr_b_cda, _ = compute_cfvr(score_fn_b_cda, test_cohort, threshold_b_cda)
bmr_b_cda = round((cfvr_b_cda - cfvr_b) / cfvr_b, 4) if cfvr_b > 0 else 0.0

print("Model B + CDA")
print(f"AUROC:    {auroc_b_cda:.4f}  (Model B baseline: {auroc_b:.4f})")
print(f"dp_ratio: {dp_ratio_b_cda:.3f}  (Model B baseline: {dp_ratio_b:.3f})")
print(f"CFVR:     {cfvr_b_cda}  (Model B baseline: {cfvr_b})")
print(f"BMR:      {bmr_b_cda:+.4f}")


Model B + CDA
AUROC:    0.6365  (Model B baseline: 0.6466)
dp_ratio: 0.000  (Model B baseline: 0.123)
CFVR:     0.0  (Model B baseline: 0.0017)
BMR:      -1.0000


## Path-Aware CDA on the Stage 1 Cohort

Same idea as the main notebook: instead of holding every non-race feature fixed when building a race-swapped twin, we resample the features that reflect unequal access rather than real health, from a real same-severity peer of the target race.

Model A uses the same pile split as the main pipeline: insurance and admission type resampled, everything else fixed. Model B sorts the 9 discharge-note features the same way: `comorbidity_burden_score`, `discharge_risk_indicator_count`, and `psychiatric_complexity` count as real health and stay fixed (psychiatric_complexity is ambiguous the same way prior_admissions_12m was on the main cohort, defaulted conservatively here instead of tested both ways to keep this bounded). The six SDOH flags go in the resampled pile with insurance and admission type, since they're exactly the kind of structural-access signal that pile is for.

Severity strata use 2 bins instead of the main notebook's 4, since this cohort is about 1% the size and finer strata would leave too few patients per race-by-stratum cell to match from.


In [46]:
N_STRATA_BINS_SCRATCH = 2

def add_severity_strata_scratch(df):
    df = df.copy()
    df["_cc_bin"]  = pd.qcut(df["comorbidity_count"], N_STRATA_BINS_SCRATCH, labels=False, duplicates="drop")
    df["_age_bin"] = pd.qcut(df["age"], N_STRATA_BINS_SCRATCH, labels=False, duplicates="drop")
    df["_stratum"] = list(zip(df["_cc_bin"], df["_age_bin"]))
    return df

def build_path_aware_twins_scratch(source_df, races, resample_cols, seed):
    rng = np.random.RandomState(seed)
    df = add_severity_strata_scratch(source_df)

    pools = {}
    for key, g in df.groupby(["_stratum", "race_clean"]):
        pools[key] = g[resample_cols].reset_index(drop=True)

    fallback_draws = 0
    total_draws    = 0
    fallback_by_race = {race: 0 for race in races}
    draws_by_race     = {race: 0 for race in races}

    twins = []
    for race in races:
        other_races = [r for r in races if r != race]
        subset = df[df["race_clean"] == race]
        for other_race in other_races:
            twin = subset.copy()
            twin["race_clean"] = other_race
            sampled_parts = []
            for stratum, idx in subset.groupby("_stratum").groups.items():
                n = len(idx)
                pool = pools.get((stratum, other_race))
                used_fallback = pool is None or len(pool) == 0
                if used_fallback:
                    pool = df.loc[df["race_clean"] == other_race, resample_cols]
                total_draws += 1
                draws_by_race[race] += 1
                if used_fallback:
                    fallback_draws += 1
                    fallback_by_race[race] += 1
                if len(pool) == 0:
                    sampled = subset.loc[idx, resample_cols].reset_index(drop=True)
                else:
                    draw = rng.randint(0, len(pool), size=n)
                    sampled = pool.iloc[draw].reset_index(drop=True)
                sampled.index = idx
                sampled_parts.append(sampled)
            sampled_all = pd.concat(sampled_parts).sort_index()
            for col in resample_cols:
                twin[col] = sampled_all[col].values
            twins.append(twin)

    twins_df = pd.concat(twins, ignore_index=True)

    fallback_rate = round(fallback_draws / total_draws, 3) if total_draws > 0 else 0.0
    print(f"severity-stratum pool empty and fell back to the unmatched race-wide pool on {fallback_draws}/{total_draws} draws ({fallback_rate:.1%})")
    for race in races:
        if draws_by_race[race] > 0:
            rate = round(fallback_by_race[race] / draws_by_race[race], 3)
            print(f"  {race:30s} fallback rate: {rate:.1%}  ({fallback_by_race[race]}/{draws_by_race[race]})")

    return twins_df.drop(columns=["_cc_bin", "_age_bin", "_stratum"])



In [47]:
PILE2_FEATURES_TAB = ["insurance_clean", "admission_type"]

twins_pace_a = build_path_aware_twins_scratch(train_cohort, RACES, PILE2_FEATURES_TAB, seed=random_seed)
train_aug_pace_a = pd.concat([train_cohort, twins_pace_a], ignore_index=True)
train_aug_pace_a["insurance_enc"]      = train_aug_pace_a["insurance_clean"].map(INS_TO_ENC).fillna(-1).astype(int)
train_aug_pace_a["admission_type_enc"] = train_aug_pace_a["admission_type"].map(ADMISSION_TO_ENC).fillna(-1).astype(int)
train_aug_pace_a["race_enc"]           = train_aug_pace_a["race_clean"].map(RACE_TO_ENC).fillna(-1).astype(int)

x_train_pace_a = train_aug_pace_a[tabular_feature_cols]
y_train_pace_a = train_aug_pace_a[target]

model_a_pace = xgb.XGBClassifier(
    **study_a.best_params,
    scale_pos_weight=(y_train_pace_a == 0).sum() / (y_train_pace_a == 1).sum(),
    random_state=random_seed,
    eval_metric="auc",
    early_stopping_rounds=20,
    verbosity=0
)
model_a_pace.fit(x_train_pace_a, y_train_pace_a, eval_set=[(x_val_a, y_val)], verbose=False)

val_preds_a_pace  = model_a_pace.predict_proba(x_val_a)[:, 1]
test_preds_a_pace = model_a_pace.predict_proba(x_test_a)[:, 1]

iso_a_pace = IsotonicRegression(out_of_bounds="clip")
iso_a_pace.fit(val_preds_a_pace, y_val)
val_cal_a_pace  = iso_a_pace.predict(val_preds_a_pace)
test_cal_a_pace = iso_a_pace.predict(test_preds_a_pace)

threshold_a_pace = pd.Series(val_cal_a_pace).quantile(0.90)
auroc_a_pace     = roc_auc_score(y_test, test_cal_a_pace)

test_cohort["enrolled_a_pace"] = (test_cal_a_pace >= threshold_a_pace).astype(int)
_, dp_ratio_a_pace = compute_fairness_metrics(
    test_cohort, "race_clean", "enrolled_a_pace", "readmitted_30d"
)

def score_fn_a_pace(df):
    return iso_a_pace.predict(model_a_pace.predict_proba(df[tabular_feature_cols])[:, 1])

cfvr_a_pace, _ = compute_cfvr(score_fn_a_pace, test_cohort, threshold_a_pace)
bmr_a_pace = round((cfvr_a_pace - cfvr_a) / cfvr_a, 4) if cfvr_a > 0 else 0.0

print("Model A + Path-Aware CDA")
print(f"AUROC:    {auroc_a_pace:.4f}  (Model A baseline: {auroc_a:.4f})")
print(f"dp_ratio: {dp_ratio_a_pace:.3f}  (Model A baseline: {dp_ratio_a:.3f})")
print(f"CFVR:     {cfvr_a_pace}  (Model A baseline: {cfvr_a})")
print(f"BMR:      {bmr_a_pace:+.4f}")


severity-stratum pool empty and fell back to the unmatched race-wide pool on 0/80 draws (0.0%)
  Asian                          fallback rate: 0.0%  (0/16)
  Black/African American         fallback rate: 0.0%  (0/16)
  Hispanic/Latino                fallback rate: 0.0%  (0/16)
  Other/Unknown                  fallback rate: 0.0%  (0/16)
  White                          fallback rate: 0.0%  (0/16)
Model A + Path-Aware CDA
AUROC:    0.6651  (Model A baseline: 0.6587)
dp_ratio: 0.110  (Model A baseline: 0.123)
CFVR:     0.0  (Model A baseline: 0.0182)
BMR:      -1.0000


In [48]:
PILE2_FEATURES_B = PILE2_FEATURES_TAB + [
    "sdoh_housing_instability", "sdoh_food_insecurity", "sdoh_substance_use",
    "sdoh_limited_social_support", "sdoh_unemployment", "sdoh_transportation_barrier",
]

twins_pace_b = build_path_aware_twins_scratch(train_cohort, RACES, PILE2_FEATURES_B, seed=random_seed)
train_aug_pace_b = pd.concat([train_cohort, twins_pace_b], ignore_index=True)
train_aug_pace_b["insurance_enc"]      = train_aug_pace_b["insurance_clean"].map(INS_TO_ENC).fillna(-1).astype(int)
train_aug_pace_b["admission_type_enc"] = train_aug_pace_b["admission_type"].map(ADMISSION_TO_ENC).fillna(-1).astype(int)
train_aug_pace_b["race_enc"]           = train_aug_pace_b["race_clean"].map(RACE_TO_ENC).fillna(-1).astype(int)

x_train_pace_b = train_aug_pace_b[all_feature_cols]
y_train_pace_b = train_aug_pace_b[target]

model_b_pace = xgb.XGBClassifier(
    **study_b.best_params,
    scale_pos_weight=(y_train_pace_b == 0).sum() / (y_train_pace_b == 1).sum(),
    random_state=random_seed,
    eval_metric="auc",
    early_stopping_rounds=20,
    verbosity=0
)
model_b_pace.fit(x_train_pace_b, y_train_pace_b, eval_set=[(x_val_b, y_val)], verbose=False)

val_preds_b_pace  = model_b_pace.predict_proba(x_val_b)[:, 1]
test_preds_b_pace = model_b_pace.predict_proba(x_test_b)[:, 1]

iso_b_pace = IsotonicRegression(out_of_bounds="clip")
iso_b_pace.fit(val_preds_b_pace, y_val)
val_cal_b_pace  = iso_b_pace.predict(val_preds_b_pace)
test_cal_b_pace = iso_b_pace.predict(test_preds_b_pace)

threshold_b_pace = pd.Series(val_cal_b_pace).quantile(0.90)
auroc_b_pace     = roc_auc_score(y_test, test_cal_b_pace)

test_cohort["enrolled_b_pace"] = (test_cal_b_pace >= threshold_b_pace).astype(int)
_, dp_ratio_b_pace = compute_fairness_metrics(
    test_cohort, "race_clean", "enrolled_b_pace", "readmitted_30d"
)

def score_fn_b_pace(df):
    return iso_b_pace.predict(model_b_pace.predict_proba(df[all_feature_cols].fillna(0))[:, 1])

cfvr_b_pace, _ = compute_cfvr(score_fn_b_pace, test_cohort, threshold_b_pace)
bmr_b_pace = round((cfvr_b_pace - cfvr_b) / cfvr_b, 4) if cfvr_b > 0 else 0.0

print("Model B + Path-Aware CDA")
print(f"AUROC:    {auroc_b_pace:.4f}  (Model B baseline: {auroc_b:.4f})")
print(f"dp_ratio: {dp_ratio_b_pace:.3f}  (Model B baseline: {dp_ratio_b:.3f})")
print(f"CFVR:     {cfvr_b_pace}  (Model B baseline: {cfvr_b})")
print(f"BMR:      {bmr_b_pace:+.4f}")


severity-stratum pool empty and fell back to the unmatched race-wide pool on 0/80 draws (0.0%)
  Asian                          fallback rate: 0.0%  (0/16)
  Black/African American         fallback rate: 0.0%  (0/16)
  Hispanic/Latino                fallback rate: 0.0%  (0/16)
  Other/Unknown                  fallback rate: 0.0%  (0/16)
  White                          fallback rate: 0.0%  (0/16)
Model B + Path-Aware CDA
AUROC:    0.6202  (Model B baseline: 0.6466)
dp_ratio: 0.200  (Model B baseline: 0.123)
CFVR:     0.0  (Model B baseline: 0.0017)
BMR:      -1.0000


In [49]:
stage1_results = pd.DataFrame([
    {"model": "A (tabular only)",        "method": "baseline",        "auroc": round(auroc_a, 4),      "dp_ratio": dp_ratio_a,      "cfvr": cfvr_a,      "bmr": 0.0},
    {"model": "A (tabular only)",        "method": "cda",             "auroc": round(auroc_a_cda, 4),  "dp_ratio": dp_ratio_a_cda,  "cfvr": cfvr_a_cda,  "bmr": bmr_a_cda},
    {"model": "A (tabular only)",        "method": "path_aware_cda",  "auroc": round(auroc_a_pace, 4), "dp_ratio": dp_ratio_a_pace, "cfvr": cfvr_a_pace, "bmr": bmr_a_pace},
    {"model": "B (tabular + LLM notes)", "method": "baseline",        "auroc": round(auroc_b, 4),      "dp_ratio": dp_ratio_b,      "cfvr": cfvr_b,      "bmr": 0.0},
    {"model": "B (tabular + LLM notes)", "method": "cda",             "auroc": round(auroc_b_cda, 4),  "dp_ratio": dp_ratio_b_cda,  "cfvr": cfvr_b_cda,  "bmr": bmr_b_cda},
    {"model": "B (tabular + LLM notes)", "method": "path_aware_cda",  "auroc": round(auroc_b_pace, 4), "dp_ratio": dp_ratio_b_pace, "cfvr": cfvr_b_pace, "bmr": bmr_b_pace},
])

print(f"STAGE 1 SUB-ANALYSIS -- {len(test_cohort)}-patient held-out test set")
print(stage1_results.to_string(index=False))

stage1_results.to_parquet("/kaggle/working/stage1_full_comparison.parquet", index=False)


STAGE 1 SUB-ANALYSIS -- 604-patient held-out test set
                  model         method  auroc  dp_ratio   cfvr  bmr
       A (tabular only)       baseline 0.6587     0.123 0.0182  0.0
       A (tabular only)            cda 0.6451     0.099 0.0000 -1.0
       A (tabular only) path_aware_cda 0.6651     0.110 0.0000 -1.0
B (tabular + LLM notes)       baseline 0.6466     0.123 0.0017  0.0
B (tabular + LLM notes)            cda 0.6365     0.000 0.0000 -1.0
B (tabular + LLM notes) path_aware_cda 0.6202     0.200 0.0000 -1.0


## MPPD: Real-Patient Check on the CDA Zero

CFVR and BMR above are measured against synthetic twins built by flipping race or resampling access-pile features. CDA's exact 0.0000 is measured against twins it built itself. Same real-patient check as the main notebook: match each non-White patient in the test set to their nearest real White neighbor on the features that reflect actual health (labs, comorbidities, comorbidity count, length of stay, ICU utilization, age), not on race, insurance, or admission type, and compare each method's score across the matched pair.

In [50]:
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

MPPD_FEATURES_A = [c for c in tabular_feature_cols if c not in
                    ["race_enc", "sex_enc", "insurance_enc", "admission_type_enc"]]
MPPD_FEATURES_B = MPPD_FEATURES_A + ["comorbidity_burden_score",
                                      "discharge_risk_indicator_count",
                                      "psychiatric_complexity_enc"]

reference_race = test_cohort["race_clean"].value_counts().idxmax()
race_arr = test_cohort["race_clean"].values
ref_pos = np.where(race_arr == reference_race)[0]

def mppd_matches_for(features):
    scaler = StandardScaler()
    x = scaler.fit_transform(test_cohort[features].values)
    nn_ref = NearestNeighbors(n_neighbors=1).fit(x[ref_pos])
    matches = {}
    for race in RACES:
        if race == reference_race:
            continue
        g_pos = np.where(race_arr == race)[0]
        if len(g_pos) == 0:
            continue
        _, nn_rel = nn_ref.kneighbors(x[g_pos])
        matches[race] = (g_pos, ref_pos[nn_rel.flatten()])
    return matches

mppd_matches_a = mppd_matches_for(MPPD_FEATURES_A)
mppd_matches_b = mppd_matches_for(MPPD_FEATURES_B)

mppd_score_fns = {
    ("A", "baseline"):       (score_fn_a,      mppd_matches_a),
    ("A", "cda"):            (score_fn_a_cda,  mppd_matches_a),
    ("A", "path_aware_cda"): (score_fn_a_pace, mppd_matches_a),
    ("B", "baseline"):       (score_fn_b,      mppd_matches_b),
    ("B", "cda"):            (score_fn_b_cda,  mppd_matches_b),
    ("B", "path_aware_cda"): (score_fn_b_pace, mppd_matches_b),
}

mppd_rows = []
for (model_name, method), (fn, matches) in mppd_score_fns.items():
    scores = fn(test_cohort)
    for race, (g_pos, matched_pos) in matches.items():
        diffs = np.abs(scores[g_pos] - scores[matched_pos])
        mppd_rows.append({"model": model_name, "method": method, "race": race,
                           "mppd": round(float(diffs.mean()), 4), "n_pairs": len(g_pos)})

mppd_df = pd.DataFrame(mppd_rows)
mppd_pivot = mppd_df.pivot_table(index=["model", "method"], columns="race", values="mppd")
print(f"MPPD, reference group: {reference_race}")
print(mppd_pivot.to_string())

mppd_df.to_parquet("/kaggle/working/stage1_mppd_results.parquet", index=False)


MPPD, reference group: White
race                   Asian  Black/African American  Hispanic/Latino  Other/Unknown
model method                                                                        
A     baseline        0.0811                  0.0972           0.0888         0.0557
      cda             0.0927                  0.0997           0.0984         0.0397
      path_aware_cda  0.0721                  0.0978           0.0977         0.0501
B     baseline        0.0665                  0.0894           0.1227         0.0590
      cda             0.0719                  0.0941           0.0913         0.0415
      path_aware_cda  0.0865                  0.0982           0.1091         0.0469


Extending the same comparison to sex and insurance, plus F1, so this table lines up with what the main notebook now reports for every method.


In [51]:
from sklearn.metrics import f1_score

model_methods = {
    ("A", "baseline"):       (score_fn_a,      threshold_a,      "enrolled_a"),
    ("A", "cda"):            (score_fn_a_cda,  threshold_a_cda,  "enrolled_a_cda"),
    ("A", "path_aware_cda"): (score_fn_a_pace, threshold_a_pace, "enrolled_a_pace"),
    ("B", "baseline"):       (score_fn_b,      threshold_b,      "enrolled_b"),
    ("B", "cda"):            (score_fn_b_cda,  threshold_b_cda,  "enrolled_b_cda"),
    ("B", "path_aware_cda"): (score_fn_b_pace, threshold_b_pace, "enrolled_b_pace"),
}

axes_cfg_stage1 = {
    "race":      ("race_clean",      "race_enc",      RACES,      RACE_TO_ENC),
    "sex":       ("sex",             "sex_enc",       SEXES,      SEX_TO_ENC),
    "insurance": ("insurance_clean", "insurance_enc", INS_GROUPS, INS_TO_ENC),
}

cfvr_baseline_by_model_axis = {}
extended_rows = []

for (model_name, method), (fn, thr, enr_col) in model_methods.items():
    f1 = round(float(f1_score(y_test, test_cohort[enr_col])), 4)
    row = {"model": model_name, "method": method, "f1": f1}
    for axis, (gcol, ecol, groups, enc_map) in axes_cfg_stage1.items():
        dp = dp_ratio_from_enrolled(test_cohort[enr_col], test_cohort[gcol])
        cfvr_ax, _ = compute_cfvr_attr(fn, test_cohort, thr, gcol, ecol, groups, enc_map)
        if method == "baseline":
            cfvr_baseline_by_model_axis[(model_name, axis)] = cfvr_ax
        base_cfvr = cfvr_baseline_by_model_axis[(model_name, axis)]
        bmr_ax = round((cfvr_ax - base_cfvr) / base_cfvr, 4) if base_cfvr > 0 else 0.0
        row[f"dp_ratio_{axis}"] = dp
        row[f"cfvr_{axis}"]     = cfvr_ax
        row[f"bmr_{axis}"]      = bmr_ax
    extended_rows.append(row)

extended_results = pd.DataFrame(extended_rows)

print(extended_results[["model", "method", "f1"]].to_string(index=False, float_format=lambda x: f"{x:.4f}"))
for axis in axes_cfg_stage1:
    print(f"\n{axis}")
    cols = ["model", "method", f"dp_ratio_{axis}", f"cfvr_{axis}", f"bmr_{axis}"]
    print(extended_results[cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

extended_results.to_parquet("/kaggle/working/stage1_extended_results.parquet", index=False)

model         method     f1
    A       baseline 0.3241
    A            cda 0.4397
    A path_aware_cda 0.3896
    B       baseline 0.3134
    B            cda 0.3900
    B path_aware_cda 0.3770

race
model         method  dp_ratio_race  cfvr_race  bmr_race
    A       baseline         0.1250     0.0182    0.0000
    A            cda         0.1000     0.0000   -1.0000
    A path_aware_cda         0.1110     0.0000   -1.0000
    B       baseline         0.1250     0.0017    0.0000
    B            cda         0.0000     0.0000   -1.0000
    B path_aware_cda         0.2000     0.0000   -1.0000

sex
model         method  dp_ratio_sex  cfvr_sex  bmr_sex
    A       baseline        0.8310    0.0215   0.0000
    A            cda        0.9710    0.0000  -1.0000
    A path_aware_cda        0.9950    0.0000  -1.0000
    B       baseline        0.9020    0.0000   0.0000
    B            cda        0.9680    0.0050   0.0000
    B path_aware_cda        0.8800    0.0000   0.0000

insurance
model

Bootstrap confidence intervals and significance on the numbers above, same 1000-resample, Benjamini-Hochberg, Cohen's h treatment as the main notebook. Race-by-model subgroups here are in the dozens, not the tens of thousands, so expect much wider intervals than the main cohort, that width is the honest picture, not a bug.


In [52]:
from scipy import stats
from statsmodels.stats.multitest import multipletests

def cohens_h(p1, p2):
    return 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))

def precompute_cf_scores_stage1(df, fn, axes_def):
    orig = fn(df)
    cf_scores = {ax: {} for ax in axes_def}
    for axis, (gcol, ecol, groups, enc_map) in axes_def.items():
        for g in groups:
            mask = (df[gcol].values != g)
            if mask.sum() == 0:
                continue
            cf = df.copy()
            cf.loc[cf.index[mask], gcol] = g
            cf.loc[cf.index[mask], ecol] = enc_map[g]
            cf_scores[axis][g] = fn(cf)
    return orig, cf_scores

def bootstrap_cfvr_stage1(orig, cf_scores, thr, df, axes_def, n_bootstrap=1000, seed=42):
    np.random.seed(seed)
    n = len(df)
    boot = {ax: np.zeros(n_bootstrap) for ax in axes_def}
    for b in range(n_bootstrap):
        idx = np.random.choice(n, n, replace=True)
        orig_enr = (orig[idx] >= thr).astype(int)
        for axis, (gcol, ecol, groups, enc_map) in axes_def.items():
            grp_idx = df[gcol].values[idx]
            any_flip = np.zeros(len(idx), dtype=int)
            for g in groups:
                if g not in cf_scores[axis]:
                    continue
                mask_g  = grp_idx != g
                cf_enr  = (cf_scores[axis][g][idx] >= thr).astype(int)
                flipped = ((cf_enr != orig_enr) & mask_g).astype(int)
                any_flip = np.maximum(any_flip, flipped)
            boot[axis][b] = any_flip.mean()
    return boot

stage1_boot = {}
for (model_name, method), (fn, thr, enr_col) in model_methods.items():
    orig, cf_scores = precompute_cf_scores_stage1(test_cohort, fn, axes_cfg_stage1)
    stage1_boot[(model_name, method)] = bootstrap_cfvr_stage1(orig, cf_scores, thr, test_cohort, axes_cfg_stage1)

stage1_stat_rows = []
for model_name in ["A", "B"]:
    base_boot = stage1_boot[(model_name, "baseline")]
    for method in ["baseline", "cda", "path_aware_cda"]:
        m_boot = stage1_boot[(model_name, method)]
        for axis in axes_cfg_stage1:
            mb = m_boot[axis]
            ci_lo, ci_hi = np.percentile(mb, [2.5, 97.5])
            if method == "baseline":
                p_val, h = float("nan"), 0.0
                diff_ci_lo = diff_ci_hi = 0.0
            else:
                bb = base_boot[axis]
                diff = bb - mb
                diff_ci_lo, diff_ci_hi = np.percentile(diff, [2.5, 97.5])
                p_val = min(2 * min((diff <= 0).mean(), (diff >= 0).mean()), 1.0)
                h = cohens_h(bb.mean(), mb.mean())
            stage1_stat_rows.append({
                "model": model_name, "method": method, "axis": axis,
                "cfvr_mean":  round(float(mb.mean()), 4),
                "ci_lo":      round(float(ci_lo), 4),
                "ci_hi":      round(float(ci_hi), 4),
                "diff_ci_lo": round(float(diff_ci_lo), 4),
                "diff_ci_hi": round(float(diff_ci_hi), 4),
                "p_value":    round(p_val, 6) if not (p_val != p_val) else float("nan"),
                "cohens_h":   round(h, 3),
            })

stage1_stat_df = pd.DataFrame(stage1_stat_rows)
mask = stage1_stat_df["method"] != "baseline"
pvals = stage1_stat_df.loc[mask, "p_value"].fillna(1.0).values
_, p_adj, _, _ = multipletests(pvals, method="fdr_bh")
stage1_stat_df.loc[mask, "p_adj"] = p_adj.round(6)

for model_name in ["A", "B"]:
    print(f"\nMODEL {model_name}")
    sub = stage1_stat_df[stage1_stat_df["model"] == model_name][
        ["method", "axis", "cfvr_mean", "ci_lo", "ci_hi", "diff_ci_lo", "diff_ci_hi", "p_adj", "cohens_h"]
    ]
    print(sub.to_string(index=False))

stage1_stat_df.to_parquet("/kaggle/working/stage1_statistical_results.parquet", index=False)



MODEL A
        method      axis  cfvr_mean  ci_lo  ci_hi  diff_ci_lo  diff_ci_hi  p_adj  cohens_h
      baseline      race     0.0180 0.0083 0.0298      0.0000      0.0000    NaN     0.000
      baseline       sex     0.0214 0.0099 0.0331      0.0000      0.0000    NaN     0.000
      baseline insurance     0.0163 0.0066 0.0281      0.0000      0.0000    NaN     0.000
           cda      race     0.0000 0.0000 0.0000      0.0083      0.0298    0.0     0.269
           cda       sex     0.0000 0.0000 0.0000      0.0099      0.0331    0.0     0.294
           cda insurance     0.0000 0.0000 0.0000      0.0066      0.0281    0.0     0.256
path_aware_cda      race     0.0000 0.0000 0.0000      0.0083      0.0298    0.0     0.269
path_aware_cda       sex     0.0000 0.0000 0.0000      0.0099      0.0331    0.0     0.294
path_aware_cda insurance     0.0000 0.0000 0.0000      0.0066      0.0281    0.0     0.256

MODEL B
        method      axis  cfvr_mean  ci_lo  ci_hi  diff_ci_lo  diff_ci_h

## Equalized Odds Gap

TPR and FPR gap by group for each method's actual enrollment decision, race, sex, and insurance. No counterfactual involved here, just the real decision each patient got.

In [53]:
def eo_gap_from_metrics(df, group_col, enrolled_col, outcome_col):
    breakdown, _ = compute_fairness_metrics(df, group_col, enrolled_col, outcome_col)
    tpr_gap = round(breakdown["tpr"].max() - breakdown["tpr"].min(), 4)
    fpr_gap = round(breakdown["fpr"].max() - breakdown["fpr"].min(), 4)
    return tpr_gap, fpr_gap

eo_gap_axes = {"race": "race_clean", "sex": "sex", "insurance": "insurance_clean"}

eo_rows = []
for (model_name, method), (fn, thr, enr_col) in model_methods.items():
    row = {"model": model_name, "method": method}
    for axis, gcol in eo_gap_axes.items():
        tpr_gap, fpr_gap = eo_gap_from_metrics(test_cohort, gcol, enr_col, "readmitted_30d")
        row[f"tpr_gap_{axis}"] = tpr_gap
        row[f"fpr_gap_{axis}"] = fpr_gap
    eo_rows.append(row)

eo_gap_df = pd.DataFrame(eo_rows)
print(eo_gap_df.to_string(index=False))

eo_gap_df.to_parquet("/kaggle/working/stage1_eo_gap_results.parquet", index=False)


model         method  tpr_gap_race  fpr_gap_race  tpr_gap_sex  fpr_gap_sex  tpr_gap_insurance  fpr_gap_insurance
    A       baseline           1.0         0.192        0.000        0.037              0.090              0.114
    A            cda           1.0         0.282        0.009        0.005              0.250              0.191
    A path_aware_cda           0.8         0.192        0.017        0.016              0.225              0.151
    B       baseline           0.8         0.235        0.013        0.015              0.233              0.087
    B            cda           0.8         0.244        0.043        0.011              0.099              0.135
    B path_aware_cda           1.0         0.211        0.056        0.036              0.087              0.183


## Calibration Error (ECE)

Same ECE definition as the main classifier notebook, computed on the held-out test set using each model's isotonic-calibrated scores, not the data the calibrator was fit on.

In [54]:
def compute_ece(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i + 1])
        if mask.sum() == 0:
            continue
        bin_acc  = y_true[mask].mean()
        bin_conf = y_prob[mask].mean()
        ece += mask.sum() * abs(bin_acc - bin_conf)
    return round(ece / len(y_true), 4)

ece_score_fns = {
    ("A", "baseline"):       score_fn_a,
    ("A", "cda"):            score_fn_a_cda,
    ("A", "path_aware_cda"): score_fn_a_pace,
    ("B", "baseline"):       score_fn_b,
    ("B", "cda"):            score_fn_b_cda,
    ("B", "path_aware_cda"): score_fn_b_pace,
}

ece_rows = []
for (model_name, method), fn in ece_score_fns.items():
    ece_val = compute_ece(y_test.values, fn(test_cohort))
    ece_rows.append({"model": model_name, "method": method, "ece": ece_val})

ece_df = pd.DataFrame(ece_rows)
print(ece_df.to_string(index=False))

ece_df.to_parquet("/kaggle/working/stage1_ece_results.parquet", index=False)


model         method    ece
    A       baseline 0.0287
    A            cda 0.0221
    A path_aware_cda 0.0206
    B       baseline 0.0169
    B            cda 0.0144
    B path_aware_cda 0.0415


## Saving results

In [55]:
test_cohort[["hadm_id", "race_clean", "readmitted_30d",
             "enrolled_a", "enrolled_b"]].to_parquet(
    "/kaggle/working/stage1_scratch_results.parquet", index=False
)

comparison.to_parquet("/kaggle/working/stage1_scratch_comparison.parquet", index=False)

print("saved stage1_scratch_results.parquet")
print("saved stage1_scratch_comparison.parquet")

saved stage1_scratch_results.parquet
saved stage1_scratch_comparison.parquet


One last check: how much Model A's raw score actually moves under a race swap before calibration and the enrollment threshold get applied. CFVR above only counts a flip once it crosses the threshold, so a model could have large raw score shifts that never show up in CFVR because none of them cross the 90th-percentile line. This is just checking that isn't happening silently.


In [56]:
orig_raw = model_a.predict_proba(test_cohort[tabular_feature_cols])[:, 1]
raw_gaps = []
for race in RACES:
    cf = test_cohort.copy()
    mask = cf["race_clean"] != race
    cf.loc[cf.index[mask], "race_clean"] = race
    cf.loc[cf.index[mask], "race_enc"] = RACE_TO_ENC[race]
    cf_raw = model_a.predict_proba(cf[tabular_feature_cols])[:, 1]
    raw_gaps.append(np.abs(cf_raw[mask.values] - orig_raw[mask.values]))
all_raw_gaps = np.concatenate(raw_gaps)
print("max RAW (pre-calibration) score shift:", all_raw_gaps.max())
print("mean RAW (pre-calibration) score shift:", all_raw_gaps.mean())

max RAW (pre-calibration) score shift: 0.056909412
mean RAW (pre-calibration) score shift: 0.0064349324
